# New Segmentation Model — DeepLabV3+ (ResNet-50)

Trains a **DeepLabV3+** model with a pretrained ResNet-50 backbone for wound segmentation.
This replaces the old vanilla U-Net and should significantly improve Dice/IoU scores.

| Model | Backbone | Input Size | Pretrained |
|-------|----------|------------|------------|
| U-Net (old) | Scratch | 256×256 | No |
| **DeepLabV3+ (new)** | ResNet-50 | 512×512 | ImageNet |

**Saves to:** `outputs/models/segmentation_deeplabv3.pth`

## Cell 1: Imports & Configuration

In [ ]:
import os, random, warnings, copy
warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.segmentation import deeplabv3_resnet50

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# --- RTX 5060 + i5-12400F hardware optimizations ---
torch.backends.cudnn.benchmark        = True  # fastest cuDNN kernels for fixed 512x512 inputs
torch.backends.cuda.matmul.allow_tf32 = True  # TF32 on Ampere/Blackwell tensor cores
torch.backends.cudnn.allow_tf32       = True

BASE         = r'C:\Users\PC\Desktop\GraduationProject\MyProjectSTILL\FirstTry'
SEG_DATA     = os.path.join(BASE, 'SegementationT+S+V')
MODEL_DIR    = os.path.join(BASE, 'outputs', 'models')
VIZ_DIR      = os.path.join(BASE, 'outputs', 'visualizations')
MODEL_PATH   = os.path.join(MODEL_DIR, 'segmentation_deeplabv3.pth')
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(VIZ_DIR, exist_ok=True)

DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE     = 512
BATCH_SIZE   = 8     # doubled vs original — uses AMP; reduce to 4 if CUDA OOM
NUM_WORKERS  = 2     # parallel data-loading workers (Windows-Jupyter safe)
EPOCHS       = 60
LR           = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE     = 12

plt.rcParams.update({
    'figure.facecolor':'#0d1117', 'axes.facecolor':'#161b22',
    'axes.edgecolor':'#30363d',   'axes.labelcolor':'#c9d1d9',
    'text.color':'#c9d1d9',       'xtick.color':'#8b949e',
    'ytick.color':'#8b949e',      'grid.color':'#21262d', 'font.size':11
})
ACCENT, GREEN, BLUE, YELLOW = '#e94560', '#3fb950', '#58a6ff', '#e3b341'

print(f'Device       : {DEVICE}')
print(f'Architecture : DeepLabV3+ / ResNet-50 backbone (ImageNet pretrained)')
print(f'Input size   : {IMG_SIZE}x{IMG_SIZE}')
print(f'Batch size   : {BATCH_SIZE}  (AMP enabled)')
print(f'Workers      : {NUM_WORKERS}')
print('Imports OK')

## Cell 2: Segmentation Dataset

In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

class SegDataset(Dataset):
    def __init__(self, root, img_size=512, augment=False):
        self.img_dir  = os.path.join(root, 'images')
        self.lbl_dir  = os.path.join(root, 'labels')
        self.img_size = img_size
        self.augment  = augment
        self.files    = sorted([f for f in os.listdir(self.img_dir)
                                if os.path.splitext(f)[1].lower() in IMG_EXTS])
        self.img_tf   = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        print(f'  Found {len(self.files)} images in {os.path.basename(root)}')

    def _load_mask(self, fname, W, H):
        stem = os.path.splitext(fname)[0]
        for ext in list(IMG_EXTS) + ['.txt']:
            c = os.path.join(self.lbl_dir, stem + ext)
            if os.path.exists(c):
                if c.endswith('.txt'):
                    mask = np.zeros((H, W), dtype=np.float32)
                    with open(c) as f:
                        for line in f:
                            p = line.strip().split()
                            if len(p) < 5:
                                continue
                            cx, cy, bw, bh = float(p[1]), float(p[2]), float(p[3]), float(p[4])
                            x1, x2 = int((cx - bw/2)*W), int((cx + bw/2)*W)
                            y1, y2 = int((cy - bh/2)*H), int((cy + bh/2)*H)
                            mask[max(0,y1):min(H,y2), max(0,x1):min(W,x2)] = 1.0
                    return mask
                else:
                    return np.array(Image.open(c).convert('L').resize((W, H), Image.NEAREST)) / 255.0
        return np.zeros((H, W), dtype=np.float32)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img   = Image.open(os.path.join(self.img_dir, fname)).convert('RGB')
        W, H  = img.size
        mask  = self._load_mask(fname, W, H)
        mask  = np.array(Image.fromarray((mask*255).astype(np.uint8)).resize(
                    (self.img_size, self.img_size), Image.NEAREST)) / 255.0

        if self.augment:
            if random.random() > 0.5:
                img  = img.transpose(Image.FLIP_LEFT_RIGHT)
                mask = np.fliplr(mask).copy()
            if random.random() > 0.5:
                img  = img.transpose(Image.FLIP_TOP_BOTTOM)
                mask = np.flipud(mask).copy()
            angle = random.choice([0, 90, 180, 270])
            if angle:
                img  = img.rotate(angle)
                mask = np.rot90(mask, k=angle//90).copy()
            if random.random() > 0.5:
                factor = random.uniform(0.7, 1.3)
                from torchvision.transforms.functional import adjust_brightness
                img = adjust_brightness(img, factor)

        img_t  = self.img_tf(img)
        mask_t = torch.from_numpy(mask).float().unsqueeze(0)
        return img_t, mask_t

print('Dataset class defined.')
print('Creating splits...')
train_ds = SegDataset(os.path.join(SEG_DATA, 'train'),      IMG_SIZE, augment=True)
val_ds   = SegDataset(os.path.join(SEG_DATA, 'validation'), IMG_SIZE, augment=False)
test_ds  = SegDataset(os.path.join(SEG_DATA, 'test'),       IMG_SIZE, augment=False)

_use_workers = NUM_WORKERS > 0
_dl_kw = dict(
    num_workers      = NUM_WORKERS,
    pin_memory       = (DEVICE.type == 'cuda'),   # faster CPU→GPU transfers
    persistent_workers = _use_workers,            # keep workers alive between epochs
    prefetch_factor  = 2 if _use_workers else None,
)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  **_dl_kw)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, **_dl_kw)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, **_dl_kw)

print(f'\nTrain: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

## Cell 3: DeepLabV3+ Model (Binary Segmentation)

In [ ]:
def build_deeplabv3_binary():
    """DeepLabV3+ with ResNet-50, adapted for binary wound segmentation."""
    model = deeplabv3_resnet50(weights='DEFAULT')
    # Replace final 1x1 conv: 256 channels -> 1 (binary)
    model.classifier[4]     = nn.Conv2d(256, 1, kernel_size=1)
    model.aux_classifier[4] = nn.Conv2d(256, 1, kernel_size=1)
    return model

model = build_deeplabv3_binary().to(DEVICE)

# torch.compile gives ~10-30% speedup on PyTorch 2.0+ (Triton-based fusion)
if hasattr(torch, 'compile') and DEVICE.type == 'cuda':
    try:
        model = torch.compile(model, mode='reduce-overhead')
        print('torch.compile enabled  (reduce-overhead mode)')
    except Exception as e:
        print(f'torch.compile skipped: {e}')

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('Model: DeepLabV3+ (ResNet-50)')
print(f'Total params     : {total_p:,}')
print(f'Trainable params : {trainable_p:,}')
print('Output shape     : (B, 1, H, W)  — sigmoid -> binary mask')

## Cell 4: Loss Function & Metrics

In [ ]:
class DiceBCELoss(nn.Module):
    """Combined Dice + BCE loss for segmentation."""
    def __init__(self, bce_weight=0.4, smooth=1.0):
        super().__init__()
        self.bce_w  = bce_weight
        self.smooth = smooth
        self.bce    = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        bce   = self.bce(logits, targets)
        probs = torch.sigmoid(logits)
        inter = (probs * targets).sum(dim=(2, 3))
        dice  = 1 - (2*inter + self.smooth) / (
                    probs.sum(dim=(2,3)) + targets.sum(dim=(2,3)) + self.smooth)
        return self.bce_w * bce + (1 - self.bce_w) * dice.mean()


def seg_metrics(logits, masks, thr=0.5):
    preds = (torch.sigmoid(logits) > thr).float()
    tp = (preds * masks).sum().item()
    fp = (preds * (1 - masks)).sum().item()
    fn = ((1 - preds) * masks).sum().item()
    tn = ((1 - preds) * (1 - masks)).sum().item()
    return {
        'dice'     : (2*tp) / (2*tp + fp + fn + 1e-8),
        'iou'      : tp / (tp + fp + fn + 1e-8),
        'pixel_acc': (tp + tn) / (tp + fp + fn + tn + 1e-8),
    }

print('DiceBCELoss and metrics defined.')

## Cell 5: Training Loop

In [ ]:
criterion  = DiceBCELoss(bce_weight=0.4)
optimizer  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# AMP scaler — FP16 mixed precision on RTX 5060 (2-3x faster, ~50% less VRAM)
_amp_enabled = (DEVICE.type == 'cuda')
scaler       = torch.cuda.amp.GradScaler(enabled=_amp_enabled)

history         = {'train_loss': [], 'val_loss': [], 'val_dice': [], 'val_iou': []}
best_dice       = 0.0
patience_ctr    = 0

print(f'AMP (mixed precision) : {"ON" if _amp_enabled else "OFF"}')
print('Starting DeepLabV3+ training...')
print(f'{"Epoch":>5} {"Train Loss":>12} {"Val Loss":>10} {"Val Dice":>10} {"Val IoU":>9}')
print('-' * 55)

for epoch in range(1, EPOCHS + 1):
    # ---- Train ----
    model.train()
    t_loss = 0.0
    for imgs, masks in train_dl:
        imgs, masks = imgs.to(DEVICE, non_blocking=True), masks.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)  # faster than zero_grad()

        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=_amp_enabled):
            out       = model(imgs)
            main_loss = criterion(out['out'], masks)
            aux_loss  = criterion(out['aux'], masks)
            loss      = main_loss + 0.4 * aux_loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)                                  # unscale before clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        t_loss += loss.item()
    t_loss /= len(train_dl)

    # ---- Validate ----
    model.eval()
    v_loss, v_dice, v_iou = 0.0, 0.0, 0.0
    with torch.no_grad():
        for imgs, masks in val_dl:
            imgs, masks = imgs.to(DEVICE, non_blocking=True), masks.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=_amp_enabled):
                out = model(imgs)
            v_loss += criterion(out['out'].float(), masks).item()
            m       = seg_metrics(out['out'].float(), masks)
            v_dice += m['dice']
            v_iou  += m['iou']
    v_loss /= len(val_dl)
    v_dice /= len(val_dl)
    v_iou  /= len(val_dl)

    scheduler.step()
    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['val_dice'].append(v_dice)
    history['val_iou'].append(v_iou)

    flag = ''
    if v_dice > best_dice:
        best_dice    = v_dice
        patience_ctr = 0
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'val_dice': v_dice, 'val_iou': v_iou}, MODEL_PATH)
        flag = ' *'
    else:
        patience_ctr += 1

    if epoch % 5 == 0 or flag:
        print(f'{epoch:>5} {t_loss:>12.4f} {v_loss:>10.4f} {v_dice:>10.4f} {v_iou:>9.4f}{flag}')

    if patience_ctr >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}  (best Dice={best_dice:.4f})')
        break

print(f'\nBest Val Dice : {best_dice:.4f}')
print(f'Model saved   : {MODEL_PATH}')

## Cell 6: Plot Training History

In [ ]:
ep = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor='#0d1117')
fig.suptitle('DeepLabV3+ Training History', color='#c9d1d9', fontsize=14, y=1.01)

for ax, title, series, cols in zip(
        axes,
        ['Loss', 'Dice Score', 'IoU'],
        [[history['train_loss'], history['val_loss']],
         [history['val_dice']],
         [history['val_iou']]],
        [[BLUE, YELLOW], [GREEN], [ACCENT]]):
    ax.set_facecolor('#161b22')
    labels = ['Train', 'Val'] if len(series) == 2 else [title]
    for s, c, l in zip(series, cols, labels):
        ax.plot(ep, s, color=c, linewidth=2, label=l)
    ax.set_title(title, color='#c9d1d9')
    ax.legend(facecolor='#21262d', labelcolor='#c9d1d9')
    ax.spines[:].set_edgecolor('#30363d')

plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, 'seg_deeplabv3_training.png'), dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved -> outputs/visualizations/seg_deeplabv3_training.png')

## Cell 7: Test Evaluation & Comparison with Old U-Net

In [ ]:
# Load best checkpoint
ck = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ck['model_state_dict'])
model.eval()
print(f'Loaded best model (epoch {ck["epoch"]}, Val Dice={ck["val_dice"]:.4f})')

test_dice, test_iou, test_acc = [], [], []
with torch.no_grad():
    for imgs, masks in test_dl:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        out = model(imgs)
        m   = seg_metrics(out['out'], masks)
        test_dice.append(m['dice'])
        test_iou.append(m['iou'])
        test_acc.append(m['pixel_acc'])

new_metrics = {
    'Dice Score'    : np.mean(test_dice),
    'IoU'           : np.mean(test_iou),
    'Pixel Accuracy': np.mean(test_acc),
}

# Old U-Net metrics from notebook 05
old_metrics = {
    'Dice Score'    : 0.6047,
    'IoU'           : 0.4337,
    'Pixel Accuracy': 0.9100,
}

print(f'\n{"="*60}')
print(f'  TEST SET COMPARISON')
print(f'{"="*60}')
print(f'  {"Metric":<18} {"U-Net (old)":>14} {"DeepLabV3+":>12} {"Delta":>9}')
print(f'{"- "*30}')
for k in new_metrics:
    o, n = old_metrics[k], new_metrics[k]
    sign = "+" if n >= o else ""
    print(f'  {k:<18} {o:>14.4f} {n:>12.4f} {sign+f"{n-o:.4f}":>9}')
print(f'{"="*60}')

## Cell 8: Visualize Predictions

In [ ]:
def denorm(t):
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    return (t.permute(1,2,0).numpy() * std + mean).clip(0, 1)

model.eval()
idxs = random.sample(range(len(test_ds)), min(4, len(test_ds)))
fig, axes = plt.subplots(len(idxs), 3, figsize=(12, 4*len(idxs)), facecolor='#0d1117')
fig.suptitle('DeepLabV3+ — Test Predictions', color='#c9d1d9', fontsize=14)

for row, idx in enumerate(idxs):
    img_t, mask_t = test_ds[idx]
    with torch.no_grad():
        out  = model(img_t.unsqueeze(0).to(DEVICE))
        pred = torch.sigmoid(out['out']).squeeze().cpu().numpy()

    p_bin = (pred > 0.5).astype(np.float32)
    m_bin = mask_t.squeeze().numpy()
    tp    = (p_bin * m_bin).sum()
    dice  = 2*tp / (p_bin.sum() + m_bin.sum() + 1e-8)

    for ax in axes[row]: ax.set_facecolor('#161b22'); ax.axis('off')
    axes[row,0].imshow(denorm(img_t)); axes[row,0].set_title('Image', color='#c9d1d9')
    axes[row,1].imshow(m_bin, cmap='gray'); axes[row,1].set_title('Ground Truth', color='#c9d1d9')
    axes[row,2].imshow(pred, cmap='gray'); axes[row,2].set_title(f'Predicted  Dice={dice:.3f}', color='#c9d1d9')

plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, 'seg_deeplabv3_predictions.png'), dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved -> outputs/visualizations/seg_deeplabv3_predictions.png')